<a href="https://colab.research.google.com/github/zhangyingchengqi/Modern-Computer-Vision-with-PyTorch/blob/master/Chapter02/Sequential_method_to_build_a_neural_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 第二章 第九节 使用序贯方法构建神经网络

Sequential类来构建复杂网络

In [1]:
x = [[1,2],[3,4],[5,6],[7,8]]
y = [[3],[7],[11],[15]]  #请观察这个数据集， 它的预计效果是输入的样本特征之和  为预测结果. 所以要求我们找到这些一些参数，让我们的神经网络能完成将输入样本特征的和 作为结果输出

In [2]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:
class MyDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x).float().to(device)
        self.y = torch.tensor(y).float().to(device)
    def __getitem__(self, ix):
        return self.x[ix], self.y[ix]
    def __len__(self):
        return len(self.x)

In [4]:
ds = MyDataset(x, y)
dl = DataLoader(ds, batch_size=2, shuffle=True)

第一层：线性变换 𝑦=xW^T+b，输入大小 2，输出大小 8

第二层：ReLU 激活，增加非线性能力

第三层：线性变换，8 → 1（回归或二分类都可能用）

In [8]:
nn.Sequential(
    nn.Linear(2, 8),  # 输入层: 输入特征维度 2, 输出 8 个特征
    nn.ReLU(),        # 激活函数: ReLU，将负数置 0
    nn.Linear(8, 1)   # 输出层: 输入 8 个特征, 输出 1 个值
).to(device)

Sequential(
  (0): Linear(in_features=2, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=1, bias=True)
)

! 表示在 Jupyter Notebook 或类似环境中执行 shell 命令（不是 Python 语法的一部分）。

这条命令会从 PyPI 安装 torch_summary 库，它可以用来可视化 PyTorch 模型结构，类似于 Keras 的 model.summary()

这个 summary() 函数可以显示：

1. 模型每一层的类型

2. 输入 / 输出张量形状

3. 参数数量（可训练参数 + 非可训练参数）

4. 总参数量

In [9]:
!pip install torch_summary
from torchsummary import summary

In [11]:
summary(model, torch.zeros(1,2));

Layer (type:depth-idx)                   Output Shape              Param #
├─Linear: 1-1                            [-1, 8]                   24
├─ReLU: 1-2                              [-1, 8]                   --
├─Linear: 1-3                            [-1, 1]                   9
Total params: 33
Trainable params: 33
Non-trainable params: 0
Total mult-adds (M): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00


## 请借用AI 对上面输出的进行解析  ----


In [12]:
loss_func = nn.MSELoss()
from torch.optim import SGD
opt = SGD(model.parameters(), lr = 0.001)
import time
loss_history = []
start = time.time()
for _ in range(50):
    for ix, iy in dl:
        opt.zero_grad()
        loss_value = loss_func(model(ix),iy)
        loss_value.backward()
        opt.step()
        loss_history.append(loss_value)
end = time.time()
print(end - start)

0.09902667999267578


In [13]:
# 以上模型已经训练好了， 下面使用验证数据集进行验证.
val = [[8,9],[10,11],[1.5,2.5]]
val = torch.tensor(val).float()   # 转为张量

In [ ]:
model(val.to(device))  # 存入设备后完成预测.

tensor([[16.7953],
        [20.6512],
        [ 4.2647]], device='cuda:0', grad_fn=<AddmmBackward>)

In [16]:
val.sum(-1) # # 计算验证数据每个样本的所有特征之和（对最后一个维度求和）

# 从这个结果来看， 输出接近于预期的结果.

tensor([17., 21.,  4.])

# 第二章 第十节 保存并加载模型   save_and_load_pytorch_model